In [ ]:
import pandas as pd
from sklearn.datasets import make_classification
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml import Pipeline

In [2]:
spark = SparkSession.builder.master("local[*]").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/26 13:00:57 WARN Utils: Your hostname, Kutays-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.101 instead (on interface en6)
26/03/26 13:00:57 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/26 13:00:57 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/26 13:00:58 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [9]:
X, y = make_classification(n_samples=2000, n_features=2, n_redundant=0, n_classes=3,n_clusters_per_class=1, random_state=42)

In [10]:
df = pd.DataFrame(X, columns=["f1","f2"])
df["label"] = y
spark_df = spark.createDataFrame(df)
spark_df.show(5)

+-------------------+-------------------+-----+
|                 f1|                 f2|label|
+-------------------+-------------------+-----+
| 1.2648423922013208| 1.3228160213985396|    1|
|-0.9374099373472848|-1.2094411771232294|    2|
|-0.8019643372260585| -1.518092438735688|    2|
| 1.6549852789952315|  1.612971013835367|    1|
| 1.7508176600569016|-1.1628205972011278|    0|
+-------------------+-------------------+-----+
only showing top 5 rows


In [11]:
train_df , test_df = spark_df.randomSplit([0.8,0.2], seed=42)

assembler = VectorAssembler(inputCols=["f1","f2"], outputCol="features")
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42, maxDepth=4, numTrees=50)

pipeline = Pipeline(stages=[assembler,rf])
pipeline_model = pipeline.fit(train_df)

In [12]:
predictions = pipeline_model.transform(test_df)
predictions.show(5)

+-------------------+-------------------+-----+--------------------+--------------------+--------------------+----------+
|                 f1|                 f2|label|            features|       rawPrediction|         probability|prediction|
+-------------------+-------------------+-----+--------------------+--------------------+--------------------+----------+
| -1.620520456874001|0.17958128271073992|    2|[-1.6205204568740...|[0.45765660856347...|[0.00915313217126...|       2.0|
| -1.601798421647545| 0.4414125553111359|    2|[-1.6017984216475...|[0.49809501515531...|[0.00996190030310...|       2.0|
| -1.569746193439071|-0.5448354446400705|    2|[-1.5697461934390...|[0.45765660856347...|[0.00915313217126...|       2.0|
|-1.4986963994826992|-0.1342241603615777|    2|[-1.4986963994826...|[0.45765660856347...|[0.00915313217126...|       2.0|
|-1.3548128974977278|-0.8877763347503504|    2|[-1.3548128974977...|[0.42270701181952...|[0.00845414023639...|       2.0|
+-------------------+---

In [13]:
evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

acc = evaluator.evaluate(predictions)

print(f"RF spark accuracy: {acc}")

RF spark accuracy: 0.9282051282051282


26/03/27 01:43:52 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 922195 ms exceeds timeout 120000 ms
26/03/27 01:43:53 WARN SparkContext: Killing executors is not supported by current scheduler.
26/03/27 02:01:43 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$